# Shortest Path Between Algerian Cities

This notebook demonstrates how to find and visualize the shortest path between Algerian cities using multiple graph search algorithms.

**Algorithms covered:**
- Dijkstra's Algorithm
- A* Search
- Breadth-First Search (BFS)
- Bellman-Ford Algorithm

In [ ]:
import sys
sys.path.insert(0, '..')

from src.cities import ALGERIAN_CITIES, list_cities
from src.graph_builder import GraphBuilder
from src.algorithms import PathFinder, format_comparison_table
from src.visualizer import RouteVisualizer
from src.benchmarks import AlgorithmBenchmark
from src.heuristics import haversine

print(f"Loaded {len(ALGERIAN_CITIES)} Algerian cities")

## 1. Build the Road Network Graph

We build a synthetic road network graph connecting Algerian cities based on their geographical proximity. Edge weights represent estimated road distances (Haversine distance multiplied by a road factor of 1.35).

In [ ]:
builder = GraphBuilder()
graph = builder.build_synthetic(ALGERIAN_CITIES)
stats = builder.get_stats()

print("Graph Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 2. Visualize the Road Network

Display all cities and road connections on an interactive map.

In [ ]:
viz = RouteVisualizer(center_lat=33.0, center_lon=3.0, zoom=5)
viz.add_cities(ALGERIAN_CITIES)
viz.add_graph_edges(graph, ALGERIAN_CITIES)
viz.get_map()

## 3. Find Shortest Path: Oran to Constantine

Let's find the shortest route from Oran (west) to Constantine (east) using A* search with the Haversine heuristic.

In [ ]:
finder = PathFinder(graph)

result = finder.astar("Oran", "Constantine", heuristic="haversine")
print(result.summary())
print()
print("Route:")
for i, city in enumerate(result.path):
    if i < len(result.path) - 1:
        next_city = result.path[i + 1]
        seg_dist = haversine(ALGERIAN_CITIES[city], ALGERIAN_CITIES[next_city]) * 1.35
        print(f"  {city} --({seg_dist:.1f} km)--> {next_city}")

# Show on map
viz2 = RouteVisualizer()
viz2.add_cities(ALGERIAN_CITIES)
viz2.add_graph_edges(graph, ALGERIAN_CITIES)
viz2.add_route(result.path, ALGERIAN_CITIES, color="#2196F3",
               label=f"A*: {result.distance_km:.1f} km")
viz2.get_map()

## 4. Compare All Algorithms

Run Dijkstra, A*, BFS, and Bellman-Ford on the same route and compare their performance.

In [ ]:
results = finder.compare_all("Oran", "Constantine")

print(format_comparison_table(results))

## 5. Performance Benchmark

Benchmark all algorithms across multiple city pairs.

In [ ]:
benchmark = AlgorithmBenchmark(graph)

city_pairs = [
    ("Oran", "Constantine"),
    ("Algiers", "Tamanrasset"),
    ("Tlemcen", "Annaba"),
    ("Ghardaia", "Béjaïa"),
    ("Adrar", "Tébessa"),
]

entries = benchmark.run_benchmark(city_pairs)
report = benchmark.generate_report()
print(report)

## 6. Benchmark Chart

In [ ]:
fig = benchmark.plot_comparison(output_path="../results/benchmark_chart.png")

## 7. Interactive Map with All Routes

Overlay routes from all four algorithms on a single map with a comparison legend.

In [ ]:
results = finder.compare_all("Oran", "Constantine")

viz3 = RouteVisualizer()
viz3.add_cities(ALGERIAN_CITIES)
viz3.add_graph_edges(graph, ALGERIAN_CITIES)
viz3.add_multiple_routes(results, ALGERIAN_CITIES)
viz3.add_comparison_legend(results)
viz3.save("../results/comparison_map.html")

print("Map saved to results/comparison_map.html")
viz3.get_map()

## 8. Conclusion

**Key findings:**

- **Dijkstra** and **A*** both find the optimal shortest path on graphs with non-negative weights. A* is typically faster because its heuristic guides the search towards the target.
- **BFS** finds the path with the fewest hops (edges), which is not necessarily the shortest in terms of distance. It is the fastest in terms of computation since it ignores edge weights.
- **Bellman-Ford** also finds the optimal path and can handle negative edge weights, but is significantly slower due to its O(V*E) time complexity.

For real-world routing between Algerian cities, **A*** with the Haversine heuristic is the best choice: it guarantees optimality while exploring fewer nodes than Dijkstra.